# Анализ лояльности пользователей Яндекс Афиши

**Цель проекта**

Проанализировать поведение пользователей сервиса и выявить факторы, влияющие на совершение повторных покупок и общее количество заказов, с целью повышения удержания пользователей и увеличения выручки.

**Задачи:**
- подготовить и очистить данные
- провести исследовательский анализ
- изучить факторы возвращаемости
- сегментировать пользователей
- выполнить корреляционный анализ
- сформулировать выводы и рекомендации

## Этапы выполнения проекта

### 1. Загрузка данных и их предобработка

---

**Задача 1.1:** Напишите SQL-запрос, выгружающий в датафрейм pandas необходимые данные. Используйте следующие параметры для подключения к базе данных `data-analyst-afisha`:

- **Хост** — `rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net`
- **База данных** — `data-analyst-afisha`
- **Порт** — `6432`
- **Аутентификация** — `Database Native`
- **Пользователь** — `praktikum_student`
- **Пароль** — `Sdf4$2;d-d30pp`

Для выгрузки используйте запрос из предыдущего урока и библиотеку SQLAlchemy.

Выгрузка из базы данных SQL должна позволить собрать следующие данные:

- `user_id` — уникальный идентификатор пользователя, совершившего заказ;
- `device_type_canonical` — тип устройства, с которого был оформлен заказ (`mobile` — мобильные устройства, `desktop` — стационарные);
- `order_id` — уникальный идентификатор заказа;
- `order_dt` — дата создания заказа (используйте данные `created_dt_msk`);
- `order_ts` — дата и время создания заказа (используйте данные `created_ts_msk`);
- `currency_code` — валюта оплаты;
- `revenue` — выручка от заказа;
- `tickets_count` — количество купленных билетов;
- `days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- `event_id` — уникальный идентификатор мероприятия;
- `service_name` — название билетного оператора;
- `event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);
- `region_name` — название региона, в котором прошло мероприятие;
- `city_name` — название города, в котором прошло мероприятие.

---


In [1]:
!pip install sqlalchemy

In [2]:
!pip install psycopg2-binary 

In [3]:
!pip install phik

In [4]:
!pip install python-dotenv

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine 
from phik import phik_matrix
import seaborn as sns
from dotenv import load_dotenv
import os

ModuleNotFoundError: No module named 'scipy.stats._mvn'

In [ ]:
load_dotenv()

db_config = {'user': os.getenv('DB_USER'), # имя пользователя
             'pwd': os.getenv('DB_PASSWORD'), # пароль
             'host': os.getenv('DB_HOST'),
             'port': os.getenv('DB_PORT'), # порт подключения
             'db': os.getenv('DB_NAME') # название базы данных
             } 

In [ ]:
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db'],
) 

In [ ]:
engine = create_engine(connection_string) 

In [ ]:
query = '''
SELECT 
  p.user_id,
  p.device_type_canonical,
  p.order_id,
  p.created_dt_msk AS order_dt,
  p.created_ts_msk AS order_ts,
  p.currency_code,
  p.revenue,
  p.tickets_count,
  EXTRACT(DAY FROM (p.created_dt_msk - LAG(p.created_dt_msk) OVER(PARTITION BY p.user_id ORDER BY p.created_dt_msk))) AS days_since_prev,
  p.event_id,
  e.event_name_code AS event_name,
  e.event_type_main,
  p.service_name,
  r.region_name,
  c.city_name
FROM afisha.purchases AS p
LEFT JOIN afisha.events AS e USING(event_id)
LEFT JOIN afisha.city AS c USING(city_id)
LEFT JOIN afisha.regions AS r USING(region_id)
WHERE (p.device_type_canonical = 'mobile' OR p.device_type_canonical = 'desktop') AND e.event_type_main <> 'фильм'
ORDER BY user_id
''' 

In [ ]:
df = pd.read_sql_query(query, con=engine) 

---

**Задача 1.2:** Изучите общую информацию о выгруженных данных. Оцените корректность выгрузки и объём полученных данных.

Предположите, какие шаги необходимо сделать на стадии предобработки данных — например, скорректировать типы данных.

Зафиксируйте основную информацию о данных в кратком промежуточном выводе.

---

In [ ]:
df.head(10)

In [ ]:
df.info()

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Типы данных нет необходимости менять, они соответствуют данным.   
Таблиц а с данными полная, пропуски только в столбце `days_since_prev`, что обозначает, что клиент совершил только один заказ.
</div>

---

###  2. Предобработка данных

Выполните все стандартные действия по предобработке данных:

---

**Задача 2.1:** Данные о выручке сервиса представлены в российских рублях и казахстанских тенге. Приведите выручку к единой валюте — российскому рублю.

Для этого используйте датасет с информацией о курсе казахстанского тенге по отношению к российскому рублю за 2024 год — `final_tickets_tenge_df.csv`. Его можно загрузить по пути `https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')`

Значения в рублях представлено для 100 тенге.

Результаты преобразования сохраните в новый столбец `revenue_rub`.

---


In [ ]:
df_rub = pd.read_csv('https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')

In [ ]:
df_rub #ппосмотр данных о курсе

In [ ]:
df_rub['data'] = df_rub['data'].astype('datetime64[ns]') #тип данных приведен к дате

In [ ]:
df = df.merge(df_rub, left_on = 'order_dt', right_on = 'data', how = 'left') #объединение таблиц по датам

In [ ]:
df['currency_code'].unique()

In [ ]:
mask = df['currency_code'] == 'kzt'
df['revenue_rub'] = df['revenue']
df.loc[mask, 'revenue_rub'] = (
    df.loc[mask, 'revenue'] / df.loc[mask, 'nominal'] * df.loc[mask, 'curs']
)

In [ ]:
df[df['currency_code'] == 'kzt'] #проверка подсчета нового столбца

In [ ]:
df = df.drop(columns=['data', 'nominal', 'curs', 'cdx']) #удаление лишних столбцов

In [ ]:
df

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
 С помощью объединения таблиц был добавлен новый столбец, в котором отображена сумма в рублях, если заказ изначально был в рублях, то стоимость перенеслась из `revenue`, если в тенге, то вычислено новое значение.
</div> 

---

**Задача 2.2:**

- Проверьте данные на пропущенные значения. Если выгрузка из SQL была успешной, то пропуски должны быть только в столбце `days_since_prev`.
- Преобразуйте типы данных в некоторых столбцах, если это необходимо. Обратите внимание на данные с датой и временем, а также на числовые данные, размерность которых можно сократить.
- Изучите значения в ключевых столбцах. Обработайте ошибки, если обнаружите их.
    - Проверьте, какие категории указаны в столбцах с номинальными данными. Есть ли среди категорий такие, что обозначают пропуски в данных или отсутствие информации? Проведите нормализацию данных, если это необходимо.
    - Проверьте распределение численных данных и наличие в них выбросов. Для этого используйте статистические показатели, гистограммы распределения значений или диаграммы размаха.
        
        Важные показатели в рамках поставленной задачи — это выручка с заказа (`revenue_rub`) и количество билетов в заказе (`tickets_count`), поэтому в первую очередь проверьте данные в этих столбцах.
        
        Если обнаружите выбросы в поле `revenue_rub`, то отфильтруйте значения по 99 перцентилю.

После предобработки проверьте, были ли отфильтрованы данные. Если были, то оцените, в каком объёме. Сформулируйте промежуточный вывод, зафиксировав основные действия и описания новых столбцов.

---

In [ ]:
df.info() #проверка пропусков

In [ ]:
df['event_type_main'].unique() #уникальные значения, не найдено заглушек

In [ ]:
df['revenue_rub'].describe() 

In [ ]:
boxplot = df.boxplot(column='revenue_rub',
                     vert=False, 
                     figsize=(12, 10))

boxplot.set_title('Распределение выручки от заказа')
boxplot.set_xlabel('Выручка от заказа')

plt.show()

In [ ]:
plt.hist(
    df['revenue_rub'],
    color='salmon',       
    edgecolor='black' 
)

plt.xlabel('Выручка от заказа')
plt.ylabel('Количество заказов')
plt.title('Распределение выручки от заказа по количеству заказов')

plt.grid()

In [ ]:
df = df.loc[df['revenue_rub'] < df['revenue_rub'].quantile(0.99)]
df = df.loc[df['revenue_rub'] >= 0]

In [ ]:
plt.hist(
    df['revenue_rub'],
    color='salmon',       
    edgecolor='black' 
)

plt.xlabel('Выручка от заказа')
plt.ylabel('Количество заказов')
plt.title('Распределение выручки от заказа по количеству заказов')

plt.grid()

In [ ]:
df['tickets_count'].describe()

In [ ]:
boxplot = df.boxplot(column='tickets_count',
                     vert=False, 
                     figsize=(12, 10))

boxplot.set_title('Распределение количества билетов')
boxplot.set_xlabel('Количество билетов')

plt.show()

In [ ]:
plt.hist(
    df['tickets_count'],
    color='lightgreen',       
    edgecolor='black' 
)

plt.xlabel('Количество билетов в одном заказе')
plt.ylabel('Количество заказов')
plt.title('Распределение количества билетов по количеству заказов')

plt.grid()

In [ ]:
df = df.loc[df['tickets_count'] < df['tickets_count'].quantile(0.99)]

In [ ]:
boxplot = df.boxplot(column='tickets_count',
                     vert=False, 
                     figsize=(12, 10))

boxplot.set_title('Распределение количества билетов')
boxplot.set_xlabel('Количество билетов')

plt.show()

In [ ]:
plt.hist(
    df['tickets_count'],
    color='lightgreen',       
    edgecolor='black' 
)

plt.xlabel('Количество билетов в одном заказе')
plt.ylabel('Количество заказов')
plt.title('Распределение количества билетов по количеству заказов')

plt.grid()

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Пропуски только в `days_since_prev`.   
   

В категориальных столбцах не было обнаружено заглушек.   
   
В столбцах `revenue_rub` и `tickets_count` были обнаружены значительные выбросы, также в `revenue_rub` есть отрицательные значения. Были построены диаграммы размаха и гистограммы, после данные были отфильтрованы по 99 процентилю и построены повторно диаграммы.
</div> 

---

### 3. Создание профиля пользователя

В будущем отдел маркетинга планирует создать модель для прогнозирования возврата пользователей. Поэтому сейчас они просят вас построить агрегированные признаки, описывающие поведение и профиль каждого пользователя.

---

**Задача 3.1.** Постройте профиль пользователя — для каждого пользователя найдите:

- дату первого и последнего заказа;
- устройство, с которого был сделан первый заказ;
- регион, в котором был сделан первый заказ;
- билетного партнёра, к которому обращались при первом заказе;
- жанр первого посещённого мероприятия (используйте поле `event_type_main`);
- общее количество заказов;
- средняя выручка с одного заказа в рублях;
- среднее количество билетов в заказе;
- среднее время между заказами.

После этого добавьте два бинарных признака:

- `is_two` — совершил ли пользователь 2 и более заказа;
- `is_five` — совершил ли пользователь 5 и более заказов.

**Рекомендация:** перед тем как строить профиль, отсортируйте данные по времени совершения заказа.

---


In [ ]:
first_orders = (
    df.sort_values('order_dt')
      .groupby('user_id')
      .first()
      .reset_index()
)

In [ ]:
first_orders = first_orders[['user_id', 'device_type_canonical', 'region_name', 'service_name', 'event_type_main']]

In [ ]:
first_orders

**Таблица с первыми заказами каждого клиента**

In [ ]:
user_profile = df.groupby('user_id').agg({
    'order_dt': ['min', 'max'],
    'order_id': 'count',
    'revenue_rub': 'mean',
    'tickets_count': 'mean',
    'days_since_prev': 'mean'
}).reset_index()

In [ ]:
user_profile.columns = ['user_id', 'date_start', 'date_last_order', 'orders_count', 'revenue_mean', 'tickets_count_mean', 'days_mean']

In [ ]:
user_profile = user_profile.merge(first_orders, on = 'user_id')

In [ ]:
user_profile['is_two'] = user_profile['orders_count']>=2

In [ ]:
user_profile['is_five'] = user_profile['orders_count']>=5

In [ ]:
user_profile

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Создан профиль пользователей, для этого была создана таблица с первыми заказами и сгруппированная по `user_id` таблица с нужными по заданию признаками, после этого объединили их и добавили два бинарных столбца.
</div> 

---

**Задача 3.2.** Прежде чем проводить исследовательский анализ данных и делать выводы, важно понять, с какими данными вы работаете: насколько они репрезентативны и нет ли в них аномалий.

Используя данные о профилях пользователей, рассчитайте:

- общее число пользователей в выборке;
- среднюю выручку с одного заказа;
- долю пользователей, совершивших 2 и более заказа;
- долю пользователей, совершивших 5 и более заказов.

Также изучите статистические показатели:

- по общему числу заказов;
- по среднему числу билетов в заказе;
- по среднему количеству дней между покупками.

По результатам оцените данные: достаточно ли их по объёму, есть ли аномальные значения в данных о количестве заказов и среднем количестве билетов?

Если вы найдёте аномальные значения, опишите их и примите обоснованное решение о том, как с ними поступить:

- Оставить и учитывать их при анализе?
- Отфильтровать данные по какому-то значению, например, по 95-му или 99-му перцентилю?

Если вы проведёте фильтрацию, то вычислите объём отфильтрованных данных и выведите статистические показатели по обновлённому датасету.

In [ ]:
user_profile.shape[0] #общее количество уникальных пользователей

In [ ]:
round((user_profile['revenue_mean']*user_profile['orders_count']).sum()/(user_profile['orders_count'].sum()),2) #средняя выручка с одного заказа

In [ ]:
round(user_profile['is_two'].sum()/user_profile.shape[0],4) #доля пользователей, совершивших 2 и более заказа

In [ ]:
round(user_profile['is_five'].sum()/user_profile.shape[0], 4) #доля пользователей, совершивших 5 и более заказов

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Всего у нас 21689 уникальных пользователей, этих данных вполне достаточно для анализа.   
Средняя выручка за один заказ составляет 512.34 рубля.   
Примерно 62% пользователей сделали 2 и более заказов, примерно 29% пользователей сделали 5 и более заказов.
</div> 

In [ ]:
user_profile['orders_count'].describe() #статистические показатели по общему числу заказов

In [ ]:
user_profile.groupby('orders_count').count()

In [ ]:
boxplot = user_profile.boxplot(column='orders_count',
                     vert=False)

boxplot.set_title('Распределение количества заказов')
boxplot.set_xlabel('Количество заказов')

plt.show()

In [ ]:
user_profile['orders_count'].quantile(0.95)

In [ ]:
user_profile['orders_count'].quantile(0.99)

In [ ]:
user_profile = user_profile.loc[user_profile['orders_count'] < user_profile['orders_count'].quantile(0.99)]

In [ ]:
boxplot = user_profile.boxplot(column='orders_count',
                     vert=False)

boxplot.set_title('Распределение количества заказов')
boxplot.set_xlabel('Количество заказов')

plt.show()

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Столбец с количеством заказа имеет значительные выбросы, основная часть пользователей сделала менее 10 заказов, выбросы от 10 до 150 можно считать приемлимымы. Были вычислены 99 и 95 процентили, 150 и 31 соответственно, было принято решение отфильтровать по 99, чтобы убрать неадекватные значения в несколько тысяч. В итоге было удалено 218 строк, а осталось в таблице 21471.
</div> 

In [ ]:
user_profile['tickets_count_mean'].describe()

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Ранее в исходной таблице я уже отфильтровала данные о количестве билетов в заказе по 99 процентиле, поэтому по пользователям не имеют выбросов.
</div> 

In [ ]:
user_profile['days_mean'].describe()

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Столбец с количеством дней между заказми имеет много пропущенных значения, но, как и было сказано ранее, это нормально для этого столбца. В данном случае отсутсвие значение говорит о том, что у пользователя был только один заказ (ранее выяснили, что это достаточно большая доля пользователей).   
   
Знчения варьируются от 0 до 148, что является нормой для такого показателя, пользователь может сделать несколько заказов за день, а также может на долгое время перестать делать заказы, нет необходимости фильтровать данные по этому столбцу.
</div> 

---

### 4. Исследовательский анализ данных

Следующий этап — исследование признаков, влияющих на возврат пользователей, то есть на совершение повторного заказа. Для этого используйте профили пользователей.



#### 4.1. Исследование признаков первого заказа и их связи с возвращением на платформу

Исследуйте признаки, описывающие первый заказ пользователя, и выясните, влияют ли они на вероятность возвращения пользователя.

---

**Задача 4.1.1.** Изучите распределение пользователей по признакам.

- Сгруппируйте пользователей:
    - по типу их первого мероприятия;
    - по типу устройства, с которого совершена первая покупка;
    - по региону проведения мероприятия из первого заказа;
    - по билетному оператору, продавшему билеты на первый заказ.
- Подсчитайте общее количество пользователей в каждом сегменте и их долю в разрезе каждого признака. Сегмент — это группа пользователей, объединённых определённым признаком, то есть объединённые принадлежностью к категории. Например, все клиенты, сделавшие первый заказ с мобильного телефона, — это сегмент.
- Ответьте на вопрос: равномерно ли распределены пользователи по сегментам или есть выраженные «точки входа» — сегменты с наибольшим числом пользователей?

---


In [ ]:
event_type_group = user_profile.groupby('event_type_main')['user_id'].count().sort_values(ascending=False)
event_type_group

In [ ]:
event_type_group_share = round(event_type_group/event_type_group.sum(),4).sort_values(ascending=False)
event_type_group_share

In [ ]:
event_type_group.plot(kind='bar')

plt.title('Распределение по типу первого мероприятия')
plt.xlabel('Тип первого мероприятия')
plt.ylabel('Количество пользователей')
plt.show()

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
По типу первого мероприятия больше всего пользователей, купивших первые билеты на концерты, на 2 и 3 местах другие категории и театр, в этих категориях заказов почти в два раза меньше, чем в концертах. В аутсайдерах 4 типа мероприятий:стендап, спорт, выставки и ёлки, на все четыре вместе приходится примерно 10% от общего числа.
</div> 

In [ ]:
device_type_group = user_profile.groupby('device_type_canonical')['user_id'].count()
device_type_group

In [ ]:
device_type_group_share = round(device_type_group/device_type_group.sum(),4).sort_values(ascending=False)
device_type_group_share

In [ ]:
device_type_group.plot(kind='bar')

plt.title('Распределение по типу устройства')
plt.xlabel('Тип устройства, с которого сделан первый заказ')
plt.ylabel('Количество пользователей')
plt.show()

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
По условию задачи были оставлены только два типа устройств, из них мобильные устройства значительно лидируют, примерно в 4 раза больше первых заказов было с делано с мобильной версии, чем с десктопной.
</div> 

In [ ]:
region_group = user_profile.groupby('region_name')['user_id'].count().sort_values(ascending=False)
region_group

In [ ]:
region_group_share = round(region_group/region_group.sum(),4).sort_values(ascending=False)
region_group_share

In [ ]:
plt.figure(figsize=(15, 10))
region_group.plot(kind='bar')

plt.title('Распределение по региону первого заказа')
plt.xlabel('Регион первого заказ')
plt.ylabel('Количество пользователей')
plt.show()

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
По регионам выделеяются Каменевский регион и Североярская область, только два этих региона формируют почти 50% всех первых заказов. Остальные регионы не превышают 6% от общего, основная масса имеет от 0 до 2%.
</div> 

In [ ]:
service_group = user_profile.groupby('service_name')['user_id'].count().sort_values(ascending=False)
service_group

In [ ]:
service_group_share = round(service_group/service_group.sum(),4).sort_values(ascending=False)
service_group_share

In [ ]:
plt.figure(figsize=(10, 6))
service_group.plot(kind='bar')

plt.title('Распределение по билетному оператору')
plt.xlabel('Билетный оператор, у которого сделан первый заказ')
plt.ylabel('Количество пользователей')
plt.show()

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Среди билетных операторов есть лидер "Билеты без проблем", на него приходится примерно 24% от общего числа первых заказов.   
Далее идет блок из четырех операторов: 'Мой билет', 'Лови билет!', 'Билеты в руки', 'Облачко', на каждый из них приходится около 10%.       
Еще можно выделить группу: "Весь в билетах" и "Лучшие билеты", у каждого примерно по 5%.   
Все остальные операторы имеют менее 3%.
</div> 

---

**Задача 4.1.2.** Проанализируйте возвраты пользователей:

- Для каждого сегмента вычислите долю пользователей, совершивших два и более заказа.
- Визуализируйте результат подходящим графиком. Если сегментов слишком много, то поместите на график только 10 сегментов с наибольшим количеством пользователей. Такое возможно с сегментами по региону и по билетному оператору.
- Ответьте на вопросы:
    - Какие сегменты пользователей чаще возвращаются на Яндекс Афишу?
    - Наблюдаются ли успешные «точки входа» — такие сегменты, в которых пользователи чаще совершают повторный заказ, чем в среднем по выборке?

При интерпретации результатов учитывайте размер сегментов: если в сегменте мало пользователей (например, десятки), то доли могут быть нестабильными и недостоверными, то есть показывать широкую вариацию значений.

---


In [ ]:
event_two = user_profile.groupby(['event_type_main', 'is_two'])['user_id'].count().unstack(fill_value=0)

event_two.plot(kind='bar')

plt.title('Распределение по типу мероприятия у пользователей совершивших один или несколько заказов')
plt.xlabel('Тип мероприятия первого заказа')
plt.ylabel('Количество пользователей')
plt.show()

In [ ]:
event_two['share_1'] = event_two[1] / (event_two[0] + event_two[1])

event_two['share_1'].sort_values().plot(kind='bar')

plt.title('Вероятность повторного заказа по типу мероприятия')
plt.xlabel('Тип мероприятия первого заказа')
plt.ylabel('Вероятность повторного заказа')
plt.show()

In [ ]:
event_two.sort_values(by='share_1')

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Выраженного влияния типа первого мероприятия на возврат пользователя не наблюдается. Вероятность возврата во всех сегментах варьируется от 52% до 64%.   
   
Ниже всего вероятность возврата в сегмента Ёлки и Спорт.
</div> 

In [ ]:
device_two = user_profile.groupby(['device_type_canonical', 'is_two'])['user_id'].count().unstack(fill_value=0)
device_two.plot(kind='bar')

plt.title('Распределение по типу устройства у пользователей совершивших один или несколько заказов')
plt.xlabel('Тип устройства первого заказа')
plt.ylabel('Количество пользователей')
plt.show()

In [ ]:
device_two['share_1'] = device_two[1] / (device_two[0] + device_two[1])

device_two['share_1'].plot(kind='bar')

plt.title('Вероятность повторного заказа по типу устройства')
plt.xlabel('Тип устройства первого заказа')
plt.ylabel('Вероятность повторного заказа')
plt.show()

In [ ]:
device_two

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Вероятность возврата при первом заказе с мобильной или десктопной версией практически равны.
</div> 


In [ ]:
top10_region = region_group[:10].index
top10_region

In [ ]:
two_region = user_profile.groupby(['region_name', 'is_two'])['user_id'].count().unstack(fill_value=0)
two_region.loc[top10_region].plot(kind='bar')

plt.title('Распределение по региону у пользователей совершивших один или несколько заказов')
plt.xlabel('Топ 10 регионов')
plt.ylabel('Количество пользователей')
plt.show()

In [ ]:
top10_region_is_two = two_region.loc[top10_region]

top10_region_is_two['share_1'] = top10_region_is_two[1] / (top10_region_is_two[0] + top10_region_is_two[1])

top10_region_is_two['share_1'].sort_values().plot(kind='bar')

plt.title('Вероятность повторного заказа по региону')
plt.xlabel('Топ 10 регионов')
plt.ylabel('Вероятность повторного заказа')
plt.show()

In [ ]:
top10_region_is_two.sort_values(by='share_1')

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Среди топ 10 регионов есть регионы: Озернинский край, Малиновоярский округ, Яблоневская область, в этих регионах вероятность возврата от 50 до 60%, остальные ирегионы имеют вероятность от 60 до 70%, но говорить о сильной взаимосвязи также не стоит.
</div>

In [ ]:
top10_service = service_group[:10].index
top10_service

In [ ]:
two_service = user_profile.groupby(['service_name', 'is_two'])['user_id'].count().unstack(fill_value=0)
two_service.loc[top10_service].plot(kind='bar')

plt.title('Распределение по оператору у пользователей совершивших один или несколько заказов')
plt.xlabel('Топ 10 билетных операторов')
plt.ylabel('Количество пользователей')
plt.show()

In [ ]:
top10_service_is_two = two_service.loc[top10_service]

top10_service_is_two['share_1'] = top10_service_is_two[1] / (top10_service_is_two[0] + top10_service_is_two[1])

top10_service_is_two['share_1'].sort_values().plot(kind='bar')

plt.title('Вероятность повторного заказа по оператору')
plt.xlabel('Топ 10 билетных операторов')
plt.ylabel('Вероятность повторного заказа')
plt.show()

In [ ]:
top10_service_is_two

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
При рассмотрении топ 10 билетных операторов не найдено влияния на возврат пользователя, у всех операторов процент возвращаемости от 60 до 65%.
</div>

---

**Задача 4.1.3.** Опираясь на выводы из задач выше, проверьте продуктовые гипотезы:

- **Гипотеза 1.** Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты.
- **Гипотеза 2.** В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах.

---

**Гипотеза 1**   
Тип мероприятия не оказывает существенного влияния на возврат пользователей. Спортивные мероприятия являются типом мероприятий с одним из самых низких процентов возвращаемости, концерт имеют большую вероятность.

**Гипотеза 2**   
Среди топ 10 регионов не наблюдается взаимосвязь между количеством пользователей в регионе и вероятности возврата.

---

#### 4.2. Исследование поведения пользователей через показатели выручки и состава заказа

Изучите количественные характеристики заказов пользователей, чтобы узнать среднюю выручку сервиса с заказа и количество билетов, которое пользователи обычно покупают.

Эти метрики важны не только для оценки выручки, но и для оценки вовлечённости пользователей. Возможно, пользователи с более крупными и дорогими заказами более заинтересованы в сервисе и поэтому чаще возвращаются.

---

**Задача 4.2.1.** Проследите связь между средней выручкой сервиса с заказа и повторными заказами.

- Постройте сравнительные гистограммы распределения средней выручки с билета (`avg_revenue_rub`):
    - для пользователей, совершивших один заказ;
    - для вернувшихся пользователей, совершивших 2 и более заказа.
- Ответьте на вопросы:
    - В каких диапазонах средней выручки концентрируются пользователи из каждой группы?
    - Есть ли различия между группами?

Текст на сером фоне:
    
**Рекомендация:**

1. Используйте одинаковые интервалы (`bins`) и прозрачность (`alpha`), чтобы визуально сопоставить распределения.
2. Задайте параметру `density` значение `True`, чтобы сравнивать форму распределений, даже если число пользователей в группах отличается.

---


In [ ]:
plt.figure(figsize=(12, 10))

plt.hist(user_profile[user_profile['is_two'] == 0]['revenue_mean'], 
         bins=30, color='lightgreen', density=True, 
         alpha=0.5, edgecolor='black', 
         label='1 заказ')

plt.hist(user_profile[user_profile['is_two'] == 1]['revenue_mean'], 
         bins=30, color='lightblue', density=True, 
         alpha=0.5, edgecolor='black', 
         label='2 и боолее заказов')

plt.title('Распределение средней выручки от заказа в зависимости от наличия повторных заказов')
plt.xlabel('Средний чек')
plt.ylabel('Частота')
plt.legend()
plt.show()

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Примерно в диапазоне средней выручки от 250 до 1000 рублей преобладают пользователи, которые возвращаются и делают повторные заказы. В остальных интервалах от 0 до 250 и от 1000 и далее, преобладают пользователи, совершившие только один заказ.
</div>

---

**Задача 4.2.2.** Сравните распределение по средней выручке с заказа в двух группах пользователей:

- совершившие 2–4 заказа;
- совершившие 5 и более заказов.

Ответьте на вопрос: есть ли различия по значению средней выручки с заказа между пользователями этих двух групп?

---


In [ ]:
plt.figure(figsize=(12, 10))

plt.hist(user_profile[(user_profile['is_two'] == 1) & (user_profile['is_five'] == 0)]['revenue_mean'], 
         bins=30, color='lightgreen', density=True, 
         alpha=0.5, edgecolor='black', 
         label='2–4 заказа')

plt.title('Распределение средней выручки от заказа в зависимости от наличия повторных заказов')
plt.xlabel('Средний чек')
plt.ylabel('Частота')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))

plt.hist(user_profile[user_profile['is_five'] == 1]['revenue_mean'], 
         bins=30, color='lightblue', density=True, 
         alpha=0.5, edgecolor='black', 
         label='5 и боолее заказов')

plt.title('Распределение средней выручки от заказа в зависимости от наличия повторных заказов')
plt.xlabel('Средний чек')
plt.ylabel('Частота')
plt.legend()
plt.show()

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
На графиках заметно, что распределения отличаются, для сегмента 2-4 заказов вероятность повторных заказов убывает вместе с повышением средней стоимости заказа.   
      
    
Для сегмента 5 и более заказов пиковое значение в районе 600 рублей, от него начинается убывание в обе стороны, то есть пользователи со средним чеком 450-600 рублей более склонны к совершению множества повторных покупок.
</div>

---

**Задача 4.2.3.** Проанализируйте влияние среднего количества билетов в заказе на вероятность повторной покупки.

- Изучите распределение пользователей по среднему количеству билетов в заказе (`avg_tickets_count`) и опишите основные наблюдения.
- Разделите пользователей на несколько сегментов по среднему количеству билетов в заказе:
    - от 1 до 2 билетов;
    - от 2 до 3 билетов;
    - от 3 до 5 билетов;
    - от 5 и более билетов.
- Для каждого сегмента подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы.
- Ответьте на вопросы:
    - Как распределены пользователи по сегментам — равномерно или сконцентрировано?
    - Есть ли сегменты с аномально высокой или низкой долей повторных покупок?

---

In [ ]:
plt.hist(user_profile['tickets_count_mean'], 
         bins=5, color='pink',
         edgecolor='black')

plt.title('Распределение среднего количества билетов в заказе')
plt.xlabel('Среднее число билетов')
plt.ylabel('Частота')
plt.show()

In [ ]:
user_profile = user_profile.copy()

In [ ]:
user_profile['tickets_segment'] = pd.cut(
    user_profile['tickets_count_mean'],
    bins=[1, 2, 3, 5, float('inf')],
    labels=['1-2', '2-3', '3-5', '5+'],
    right=False
)

In [ ]:
segment_stats = (user_profile.groupby('tickets_segment').agg(
        users=('user_id', 'count'),
        return_rate=('is_two', 'mean')))
segment_stats

In [ ]:
segment_stats['return_rate'].plot(kind='bar', color='pink',
         edgecolor='black')

plt.title('Распределение вероятности возврата по сегментам')
plt.ylabel('Вероятность повторного заказа')
plt.xlabel('Сегмент по среднему количеству билетов')

plt.show()

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Большая часть пользователей в среднем покупает 2-3 билета в одном заказе, именно этот сегмент пользователей имеет наиболее высокую вероятность сделать повторный заказ, более 70%. Пользователи из сегментов 1-2 и 3-5 имеют вероятность 50-60% и примерно равны, а вот пользователи со средним числом билетов более пяти имеют наименьшую вероятность возврата на платформу. 
</div>

---

#### 4.3. Исследование временных характеристик первого заказа и их влияния на повторные покупки

Изучите временные параметры, связанные с первым заказом пользователей:

- день недели первой покупки;
- время с момента первой покупки — лайфтайм;
- средний интервал между покупками пользователей с повторными заказами.

---

**Задача 4.3.1.** Проанализируйте, как день недели, в которой была совершена первая покупка, влияет на поведение пользователей.

- По данным даты первого заказа выделите день недели.
- Для каждого дня недели подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы. Результаты визуализируйте.
- Ответьте на вопрос: влияет ли день недели, в которую совершена первая покупка, на вероятность возврата клиента?

---


In [ ]:
user_profile['first_weekday'] = user_profile['date_start'].dt.day_name()

In [ ]:
weekday_two = user_profile.groupby('first_weekday')['is_two'].sum()
weekday_two

In [ ]:
weekday_all_users = user_profile.groupby('first_weekday')['user_id'].count()
weekday_all_users

In [ ]:
weekday_percent_two = user_profile.groupby('first_weekday')['is_two'].sum()/user_profile.groupby('first_weekday')['user_id'].count()
weekday_percent_two

In [ ]:
weekday_percent_two.plot(kind='bar', color='lightgreen',
         edgecolor='black')

plt.title('Вероятность возврата по дню недели')
plt.ylabel('Вероятность повторного заказа')
plt.xlabel('День недели первого заказа')

plt.show()

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Не обнаружено существенного влияния дня недели первого заказа на последующее совершение заказов. Вероятность примерно равна для всех дней недели, варьируется от 59 до 63%.
</div>

---

**Задача 4.3.2.** Изучите, как средний интервал между заказами влияет на удержание клиентов.

- Рассчитайте среднее время между заказами для двух групп пользователей:
    - совершившие 2–4 заказа;
    - совершившие 5 и более заказов.
- Исследуйте, как средний интервал между заказами влияет на вероятность повторного заказа, и сделайте выводы.

---


In [ ]:
users_2to4 = user_profile[(user_profile['is_two'] ==1) & (user_profile['is_five'] ==0)]
(users_2to4['days_mean']*users_2to4['orders_count']).sum()/users_2to4['orders_count'].sum()

In [ ]:
users_5_and_more = user_profile[user_profile['is_five'] ==1]
(users_5_and_more['days_mean']*users_5_and_more['orders_count']).sum()/users_5_and_more['orders_count'].sum()

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Пользователи с большим количеством заказов имеют меньший средний интервал между заказами. Это указывает на более высокую вовлечённость и регулярность использования сервиса.
</div>

---

#### 4.4. Корреляционный анализ количества покупок и признаков пользователя

Изучите, какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок. Для этого используйте универсальный коэффициент корреляции `phi_k`, который позволяет анализировать как числовые, так и категориальные признаки.

---

**Задача 4.4.1:** Проведите корреляционный анализ:
- Рассчитайте коэффициент корреляции `phi_k` между признаками профиля пользователя и числом заказов (`total_orders`). При необходимости используйте параметр `interval_cols` для определения интервальных данных.
- Проанализируйте полученные результаты. Если полученные значения будут близки к нулю, проверьте разброс данных в `total_orders`. Такое возможно, когда в данных преобладает одно значение: в таком случае корреляционный анализ может показать отсутствие связей. Чтобы этого избежать, выделите сегменты пользователей по полю `total_orders`, а затем повторите корреляционный анализ. Выделите такие сегменты:
    - 1 заказ;
    - от 2 до 4 заказов;
    - от 5 и выше.
- Визуализируйте результат корреляции с помощью тепловой карты.
- Ответьте на вопрос: какие признаки наиболее связаны с количеством заказов?

---

In [ ]:
user_profile_phik = user_profile.drop(['user_id', 'is_two', 'is_five', 'date_start', 'date_last_order'], axis = 1)

**Убираем столбцы, которые не нужны для корреляции, '`is_two` и `is_five` напрямую связаны с количеством заказов, 
`date_start` и `date_last_order` также имеют прямую взаимосвязь, чем больше период активности пользователя, тем больше заказов он мог успеть сделать**

In [ ]:
phik_corr = user_profile_phik.phik_matrix(interval_cols = ['orders_count', 'revenue_mean', 'tickets_count_mean', 'days_mean'])

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(phik_corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.show()

In [ ]:
phik_corr['orders_count'].sort_values(ascending=False)

**Попробуем сделать сегментацию по количествам заказов:**

In [ ]:
user_profile_phik['orders_segment'] = pd.cut(
    user_profile['orders_count'],
    bins=[0, 1, 4, float('inf')],
    labels=['1_order', '2_4_orders', '5_plus_orders']
)

In [ ]:
user_profile_phik = user_profile_phik.drop('orders_count', axis = 1)

In [ ]:
phik_corr = user_profile_phik.phik_matrix(interval_cols = ['revenue_mean', 'tickets_count_mean', 'days_mean'])

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(phik_corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Phi_k correlation matrix")
plt.show()

In [ ]:
phik_corr['orders_segment'].sort_values(ascending=False)

<div style="background-color: #d9ffbd; padding: 10px; border-left: 5px solid #337500;">
Самая сильная связь со столбцом `tickets_count_mean` (0.66). Чем больше билетов в среднем покупает пользователь, тем больше у него заказов, это логично: активные пользователи берут больше всего.   

Умеренная связь со столбцами `days_mean` (0.39), `revenue_mean` (0.33), `tickets_segment` (0.24).   
Слабая связь со столбцами `region_name` (0.12), `service_name` (0.07), `event_type_main` (0.04), `device_type_canonical` (0.02)
</div>

### 5. Общий вывод и рекомендации

В конце проекта напишите общий вывод и рекомендации: расскажите заказчику, на что нужно обратить внимание. В выводах кратко укажите:

- **Информацию о данных**, с которыми вы работали, и то, как они были подготовлены: например, расскажите о фильтрации данных, переводе тенге в рубли, фильтрации выбросов.
- **Основные результаты анализа.** Например, укажите:
    - Сколько пользователей в выборке? Как распределены пользователи по числу заказов? Какие ещё статистические показатели вы подсчитали важным во время изучения данных?
    - Какие признаки первого заказа связаны с возвратом пользователей?
    - Как связаны средняя выручка и количество билетов в заказе с вероятностью повторных покупок?
    - Какие временные характеристики влияют на удержание (день недели, интервалы между покупками)?
    - Какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок согласно результатам корреляционного анализа?
- Дополните выводы информацией, которая покажется вам важной и интересной. Следите за общим объёмом выводов — они должны быть компактными и ёмкими.

В конце предложите заказчику рекомендации о том, как именно действовать в его ситуации. Например, укажите, на какие сегменты пользователей стоит обратить внимание в первую очередь, а какие нуждаются в дополнительных маркетинговых усилиях.

- Была проведена работа с исходной таблицей, в которой было 290611 строк и 15 столбцов, данные были полными, пропуски были только в одном столбце.   
- В ходе изучения данных было найдено, что есть различия в валюте, с помощью таблицы с курсом тенге столбец со стоимостью заказа был пересчитан в рубли и добавлен новый столбец.   
- Были удалены выбросы в столбцах с выручкой и количеством былетов в заказе, отфильтровали по 99 процентилю. Далее была создана новая таблица на основе исходной, в которой содержалась информация о каждом уникальном пользователи.   
- В этой таблице было 21689 уникальных покупателей, также отфильтровали данные по 99 процентилю по количеству заказов также из-за наличия выбросов. 

Проведение исследования взимосвязи признаков первого заказа и возвращаемости пользователей
показало, что как такового сильного влияния на вероятность повторного заказа не оказывают тип мероприятия, устройства, регион и билетный оператор первого заказа.   
   
Далеее был проведен анализ взаимосвязи между показателями выручки и вероятностью повторного заказа, 
Было выявлено, что примерно в диапазоне средней выручки от 250 до 1000 рублей преобладают пользователи, которые возвращаются и делают повторные заказы. 
В остальных интервалах от 0 до 250 и от 1000 и далее, преобладают пользователи, совершившие только один заказ.   
   
Также пользователи были сегментированы по количеству заказов: 2-4 и более 5, выявлено, что распределения отличаются, для сегмента 2-4 заказов вероятность повторных 
заказов убывает вместе с повышением средней стоимости заказа.
Для сегмента 5 и более заказов пиковое значение в районе 600 рублей, от него начинается убывание в обе стороны, то есть пользователи со средним чеком 450-600 
рублей более склонны к совершению множества повторных покупок.

После проведения сегментации пользователей по среднему количеству билетов в заказе было выявлено, что Большая часть пользователей в среднем покупает 2-3 билета в 
одном заказе, именно этот сегмент пользователей имеет наиболее высокую вероятность сделать повторный заказ, более 70%. Пользователи из сегментов 1-2 и 3-5 имеют 
вероятность 50-60% и примерно равны, а вот пользователи со средним числом билетов более пяти имеют наименьшую вероятность возврата на платформу.

При анализе взаимосвязи между днем недели первого заказа и возвращаемостью пользователей на платформу также не было обнаружено сущетсвенного влияния.

Изчение среднего интервала между заказами показало, что пользователи с большим количеством заказов имеют меньший средний интервал между заказами. Это указывает на 
более высокую вовлечённость и регулярность использования сервиса.

При изучении корреляции между количеством заказов и другими признаками были сделаны выводы, что Наибольшую связь с количеством заказов показывает среднее число билетов в заказе (phi_k ≈ 0.66), что указывает на то, что более активные пользователи совершают заказы с большим числом билетов.
Также умеренная связь наблюдается со средним интервалом между заказами и средней выручкой, однако эти признаки частично производны от количества заказов.
Категориальные признаки, такие как регион, тип устройства и тип события, демонстрируют слабую связь с количеством заказов, что говорит о том, что поведение пользователей слабо зависит от этих характеристик.

**Итоговый вывод:**   
В целом проведённый анализ показывает, что ключевую роль в формировании повторных покупок играют поведенческие характеристики пользователей, а не их демографические или технические параметры. Такие признаки, как регион, тип устройства или тип мероприятия, оказывают минимальное влияние на возвращаемость пользователей и количество заказов.

Наиболее значимыми факторами являются: среднее количество билетов в заказе, средняя выручка, частота покупок (интервал между заказами).

Это говорит о том, что лояльность пользователя формируется через его опыт взаимодействия с сервисом, а не через внешние характеристики.

**Рекомендации заказчику:**
- Сфокусироваться на "ядре" пользователей, основной целевой сегмент: пользователи с 2–3 билетами и средним чеком ~400–600 рублей, именно они наиболее склонны к повторным покупкам и формируют стабильную выручку
- Стимулировать второй заказ, так как большинство пользователей совершают только один заказ, критически важно: внедрять триггерные коммуникации после первой покупки, давать персональные предложения или скидки на следующий заказ
- Работать с частотой взаимодействия, сокращение интервала между заказами увеличивает вероятность удержания. Можно использовать: напоминания, персональные рекомендации, ограниченные по времени предложения

### 6. Финализация проекта и публикация в Git

Когда вы закончите анализировать данные, оформите проект, а затем опубликуйте его.

Выполните следующие действия:

1. Создайте файл `.gitignore`. Добавьте в него все временные и чувствительные файлы, которые не должны попасть в репозиторий.
2. Сформируйте файл `requirements.txt`. Зафиксируйте все библиотеки, которые вы использовали в проекте.
3. Вынести все чувствительные данные (параметры подключения к базе) в `.env`файл.
4. Проверьте, что проект запускается и воспроизводим.
5. Загрузите проект в публичный репозиторий — например, на GitHub. Убедитесь, что все нужные файлы находятся в репозитории, исключая те, что в `.gitignore`. Ссылка на репозиторий понадобится для отправки проекта на проверку. Вставьте её в шаблон проекта в тетрадке Jupyter Notebook перед отправкой проекта на ревью.

**Вставьте ссылку на проект в этой ячейке тетрадки перед отправкой проекта на ревью.**